In [12]:
# Save country level mortality by year and for each ensemble

In [1]:
import os
import xarray as xr
import numpy as np
import dask
import dask.array as da
import warnings
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality

In [2]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [3]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [5]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

in_file = "GBD_Country_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [6]:
n_samples = 1000

# === Scalar distributions (assuming normal dist.) ===
# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

# Convert to Dask for broadcasting
tmrel_dask = da.from_array(tmrel_samples[:, np.newaxis, np.newaxis],
                           chunks=(100, 1, 1))
beta_dask = da.from_array(beta_samples[:, np.newaxis, np.newaxis],
                          chunks=(100, 1, 1))

In [7]:
def monte_carlo(n_samples, BMR, POP, O3):
    lat = BMR["lat"]
    lon = BMR["lon"]

    # === BMR ===
    bmr_m = BMR.sel(quantile="mean")
    bmr_l = BMR.sel(quantile="lower")
    bmr_u = BMR.sel(quantile="upper")

    BMR_mean = bmr_m.chunk({"lat": 180, "lon": 360})
    BMR_lower = bmr_l.chunk({"lat": 180, "lon": 360})
    BMR_upper = bmr_u.chunk({"lat": 180, "lon": 360})

    # Standard deviation for BMR
    bmr_std = (BMR_upper - BMR_lower) / (2 * 1.96)

    # Sample BMR: shape = (samples, lat, lon)
    bmr_samples = da.random.normal(
        loc=BMR_mean.data, scale=bmr_std.data,
        size=(n_samples, len(lat), len(lon)),
        chunks=(100, 180, 360))

    POP = POP.chunk({"lat": 180, "lon": 360})
    O3 = O3.chunk({"lat": 180, "lon": 360})

    # Broadcast POP and x to sample dimension
    pop_samples = da.broadcast_to(POP.data, (n_samples, len(lat), len(lon)))
    O3_samples = da.broadcast_to(O3.data, (n_samples, len(lat), len(lon)))

    return bmr_samples, pop_samples, O3_samples

In [8]:
def create_stats(da):
    mean = da.mean()
    lower = da.quantile(0.025, dim="sample")
    upper = da.quantile(0.975, dim="sample")

    combined = xr.DataArray(
        data=[mean.values, lower.values, upper.values],
        coords={"quantile": ["mean", "0.025", "0.975"]},
        dims=["quantile"],
        name="Mortality"
    )
    return combined

In [19]:
# === Path config ===
warnings.filterwarnings('ignore')
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["ARISE"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        o3_path = os.path.join(O3_DIR, o3_file)
        o3 = xr.open_dataarray(o3_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
        population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

        # Flag if any nans present (i.e. reindex was out of tolerance distance)
        assert not population.isnull().any()

        M = []

        for year in o3["year"].values:
            print(f"Processing year {year}")
            POP = population.sel(year=year)
            O3 = o3.sel(year=year)
            BMR_samples, POP_samples, O3_samples = monte_carlo(n_samples, BMR,
                                                               POP, O3)

            AF = att_frac(O3_samples, tmrel_dask, beta_dask)
            mortality_samples = mortality(AF, BMR_samples, POP_samples)
            mortality_samples = mortality_samples.astype("float32")
            mortality_year = xr.DataArray(
                mortality_samples,
                dims=("sample", "lat", "lon"),
                coords={"sample": np.arange(n_samples),
                        "lat": BMR.lat, "lon": BMR.lon},
                name="Mortality"
            ).chunk({"sample": -1, "lat": 45, "lon": 45})
            del mortality_samples, AF, BMR_samples, POP_samples, O3_samples

            sums = [mortality_year.where(country_mask.isel(country=i) == 1, drop=True).sum(dim=("lat", "lon")) 
                    for i in range(len(country_mask.country))]

            sums_computed = dask.compute(*sums)

            countries = [create_stats(s) for s in sums_computed]

            mortality_country = xr.concat(
                countries,
                dim=xr.DataArray(country_mask["country"],
                                 dims="country", name="country")
            )

            M.append(mortality_country)
            del mortality_year

        mortality_timeseries = xr.concat(
            M,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        )

        out_file = f"Mortality_country_sum_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        description = ("Country level mean [95% CI] COPD mortality due to "
                       "surface ozone - scripts by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Processing year 2035
Processing year 2036


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x154e8ff90920>>
Traceback (most recent call last):
  File "/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Processing year 2037
Processing year 2038
Processing year 2039
Processing year 2040
Processing year 2041
Processing year 2042
Processing year 2043
Processing year 2044
Processing year 2045
Processing year 2046
Processing year 2047
Processing year 2048
Processing year 2049
Processing year 2050
Processing year 2051
Processing year 2052
Processing year 2053
Processing year 2054
Processing year 2055
Processing year 2056
Processing year 2057
Processing year 2058
Processing year 2059


KeyboardInterrupt: 

In [16]:
# === Path config ===
warnings.filterwarnings('ignore')
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["ARISE"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        o3_path = os.path.join(O3_DIR, o3_file)
        o3 = xr.open_dataarray(o3_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
        population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

        # Flag if any nans present (i.e. reindex was out of tolerance distance)
        assert not population.isnull().any()

        M = []

        for year in o3["year"].values:
            print(f"Processing year {year}")
            POP = population.sel(year=year)
            O3 = o3.sel(year=year)
            BMR_samples, POP_samples, O3_samples = monte_carlo(n_samples, BMR,
                                                               POP, O3)

            AF = att_frac(O3_samples, tmrel_dask, beta_dask)
            mortality_samples = mortality(AF, BMR_samples, POP_samples)
            mortality_samples = mortality_samples.astype("float32")
            mortality_year = xr.DataArray(
                mortality_samples,
                dims=("sample", "lat", "lon"),
                coords={"sample": np.arange(n_samples),
                        "lat": BMR.lat, "lon": BMR.lon},
                name="Mortality"
            ).chunk({"sample": -1, "lat": 45, "lon": 45})
            del mortality_samples, AF, BMR_samples, POP_samples, O3_samples
            mortality_year = mortality_year.persist()

            countries = []
            # Loop over countries and sum the mortality for each country
            for i in range(len(country_mask.country)):
                print(f"Country number {i}")
                mask = country_mask.isel(country=i)
                masked = mortality_year.where(mask == 1, drop=True)
                countrysum = masked.sum(dim=("lat", "lon")).compute()
                countrysum = countrysum.drop_vars("country", errors='ignore')
                countrysum_stats = create_stats(countrysum)
                countries.append(countrysum_stats)
                del masked, countrysum

            mortality_country = xr.concat(
                countries,
                dim=xr.DataArray(country_mask["country"],
                                 dims="country", name="country")
            )

            M.append(mortality_country)
            del mortality_year

        mortality_timeseries = xr.concat(
            M,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        )

        out_file = f"Mortality_country_sum_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        description = ("Country level mean [95% CI] COPD mortality due to "
                       "surface ozone - scripts by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Processing year 2035
Country number 0
Country number 1
Country number 2
Country number 3
Country number 4
Country number 5
Country number 6
Country number 7
Country number 8
Country number 9
Country number 10
Country number 11
Country number 12
Country number 13
Country number 14
Country number 15
Country number 16
Country number 17
Country number 18
Country number 19
Country number 20
Country number 21
Country number 22
Country number 23
Country number 24
Country number 25
Country number 26
Country number 27
Country number 28
Country number 29
Country number 30
Country number 31
Country number 32
Country number 33
Country number 34
Country number 35
Country number 36
Country number 37
Country number 38
Country number 39
Country number 40
Country number 41
Country number 42
Country number 43
Country number 44
Country number 45
Country number 46
Country number 47
Country number 48
Country number 49
Country number 50
Country number 51
Country number 52
Count

KeyboardInterrupt: 

In [14]:
from pympler import asizeof

# Collect all global variables and compute their total size
sizes = []
for name, var in list(globals().items()):
    if name.startswith("__") and name.endswith("__"):
        continue  # Skip built-in entries like __name__, __file__, etc.
    try:
        size = asizeof.asizeof(var)
        sizes.append((name, size))
    except Exception as e:
        print(f"Could not measure size of {name}: {e}")

# Sort and print top 10 variables by size
print("\nTop memory-using global variables:")
for name, size in sorted(sizes, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{name}: {size / 1024 / 1024:.2f} MB")


Could not measure size of exit: invalid option: reset(base=-14288)
Could not measure size of quit: invalid option: reset(base=-14288)
Could not measure size of sys: invalid option: reset(base=-14288)
Could not measure size of BMR: invalid option: reset(base=-14288)
Could not measure size of country_mask: 'dtype'
Could not measure size of mortality_year: 'dtype'

Top memory-using global variables:
masked: 49438.57 MB
population: 9986.80 MB
POP: 5042.94 MB
O3: 1730.56 MB
o3: 1681.13 MB
mask: 12.51 MB
countries: 0.70 MB
np: 0.70 MB
da: 0.24 MB
os: 0.14 MB
